In [ ]:
!nvidia-smi
%matplotlib inline

In [1]:
import os
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from tqdm import tqdm
from typing import List, Literal, List, Dict, Any, Optional
import numpy as np
import seaborn as sns

from datasets import load_dataset
import random
import json
import re
from functools import partial
from datasets import Dataset
from copy import deepcopy
import evaluate
import nltk
from scipy.stats import ttest_ind
import string
from collections import Counter

import openai
import os
import time
import pandas as pd
import torch

from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from dotenv import load_dotenv
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## QA implementations

### Setup

In [2]:
underspecified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_frames_sample_UND.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_frames_sample_FS.jsonl",
    split="all"
)

### Implementation

In [3]:
from openai import OpenAI
google_api = os.environ.get("GOOGLE_API_KEY")
client = OpenAI(api_key=google_api, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [4]:
from helper_functions_qa import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)

In [5]:
df_UND = pd.read_json('./intermediate/BASELINE_classified_frames_sample_UND.jsonl', lines=True)
df_FS = pd.read_json('./intermediate/BASELINE_classified_frames_sample_FS.jsonl', lines=True)

In [6]:
df_UND

,id,question,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred
0,2,How many years earlier would Punxsutawney Phil...,87,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""How many years earlier would ...",underspecified
1,3,"As of August 1, 2024, which country were holde...",France,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""As of August 1, 2024, which cou...",underspecified
2,4,What is the name of the vocalist from the firs...,Jens Kidman,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What is the name of the vocal...",underspecified
3,6,I have an element in mind and would like you t...,Mendelevium is named after Dmitri Mendeleev.,"<think>\nOkay, let me try to work through this...","{\n ""query"": ""I have an element in mind and w...",underspecified
4,7,"As of Aug 3, 2024, the artist who released the...",2,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""As of Aug 3, 2024, the artist...",underspecified
...,...,...,...,...,...,...
435,812,What is the name of the father of the first co...,Peter Treschow,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What is the name of the fathe...",underspecified
436,814,Who was the winner of Tour de France the same ...,Louison Bobet,"<think>\nOkay, let's see. The user is asking f...","{\n ""query"": ""Who was the winner of Tour de F...",underspecified
437,815,A 2002 science fiction novel by an American au...,The Sea of Trolls trilogy,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""A 2002 science fiction novel ...",underspecified
438,822,Which movie musical produced a song that was i...,Fame,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Which movie musical produced a ...",underspecified


In [7]:
df_FS

,id,question,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred
0,0,If my future wife has the same first name as t...,Jane Ballou,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""If my future wife has the sam...",fully specified
1,1,Imagine there is a building called Bronte towe...,37th,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Imagine there is a building cal...",fully specified
2,5,"According to the 2000 United States census, wh...",506000,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""According to the 2000 United St...",fully specified
3,9,The Pope born Pietro Barbo ended a long-runnin...,The Battle of Hastings.,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""The Pope born Pietro Barbo ende...",fully specified
4,10,"An Australian artist, born the same year as ar...",Reve d'Or,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""An Australian artist, born th...",fully specified
...,...,...,...,...,...,...
379,817,I grew up in a village on Long Island in the T...,Sea Cliff Elementary School,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""I grew up in a village on Long ...",fully specified
380,818,Who was the Catholic Pope eleven years after E...,Eugene II,"<think>\nOkay, let's see. The user is asking, ...","{\n ""query"": ""Who was the Catholic Pope eleve...",fully specified
381,819,How many years after publishing his paper *On ...,Seven,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""How many years after publishing...",fully specified
382,820,This singer represented Sweden in Eurovision f...,Aquarius,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""This singer represented Sweden ...",fully specified


In [8]:
short_results_UND = batch_QA_with_progress(
    underspecified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_short_answer",
    fill_value=["error"],
    client=client,
    model="gemini-2.5-flash",
    temperature=0.0
)

Running model_short_answer: 100%|██████████| 44/44 [2:44:29<00:00, 224.30s/it]  


In [9]:
# batch QA for FS
short_results_FS = batch_QA_with_progress(
    fully_specified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_short_answer",
    fill_value=["error"],
    client=client,
    model="gemini-2.5-flash",
    temperature=0.0
)

Running model_short_answer: 100%|██████████| 39/39 [1:21:21<00:00, 125.17s/it]


In [10]:
qa_underspecified = deepcopy(underspecified_set)

for key in short_results_UND:
    qa_underspecified = qa_underspecified.add_column(key, short_results_UND[key])

qa_underspecified.to_json("./intermediate/BASELINE_frames_UND_qa_Gemini.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 74.46ba/s]


1955912

In [11]:
qa_fully_specified = deepcopy(fully_specified_set)

for key in short_results_FS:
    qa_fully_specified = qa_fully_specified.add_column(key, short_results_FS[key])

qa_fully_specified.to_json("./intermediate/BASELINE_frames_FS_qa_Gemini.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 89.57ba/s]


1670632

In [ ]:
df = pd.read_json("./intermediate/BASELINE_frames_UND_qa_Gemini.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_frames_UND_qa_Gemini.csv')

df = pd.read_json("./intermediate/BASELINE_frames_FS_qa_Gemini.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_frames_FS_qa_Gemini.csv')

## Evaluations

In [14]:
underspecified_set_qa = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_frames_UND_qa_Gemini.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set_qa = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_frames_FS_qa_Gemini.jsonl",
    split="all"
)

Generating train split: 440 examples [00:00, 21051.90 examples/s]
Generating train split: 384 examples [00:00, 70041.87 examples/s]


### Squad EM + F1

In [15]:
from helper_functions_qa import evaluate_squad_per_sample_multi_ref_pred

In [16]:
# Official squad script for avg EM and F1, not possible for t-test


# Evaluate fully specified subset
dataset = load_dataset("json", data_files="./intermediate/BASELINE_frames_FS_qa_Gemini.jsonl", split="all")

# 加载 HuggingFace 的 squad 评估器
squad_metric = evaluate.load("squad")

# 构造 predictions 和 references（标准格式）
predictions = [
    {
        "id": str(i),
        "prediction_text": pred[0] if isinstance(pred, list) and pred else ""
    }
    for i, pred in enumerate(dataset["model_short_answer"])
]

references = [
    {
        "id": str(i),
        "answers": {
            "text": ref if isinstance(ref, list) else [ref],
            "answer_start": [0] * len(ref if isinstance(ref, list) else [ref])
        }
    }
    for i, ref in enumerate(dataset["answer"])
]

# 计算 SQuAD-style EM 和 F1
results = squad_metric.compute(predictions=predictions, references=references)

# 打印平均指标
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")

Exact Match: 41.41
F1 Score: 57.44


In [17]:
# Official squad script for avg EM and F1, not possible for t-test


# Evaluate fully specified subset
dataset = load_dataset("json", data_files="./intermediate/BASELINE_frames_UND_qa_Gemini.jsonl", split="all")

# 加载 HuggingFace 的 squad 评估器
squad_metric = evaluate.load("squad")

# 构造 predictions 和 references（标准格式）
predictions = [
    {
        "id": str(i),
        "prediction_text": pred[0] if isinstance(pred, list) and pred else ""
    }
    for i, pred in enumerate(dataset["model_short_answer"])
]

references = [
    {
        "id": str(i),
        "answers": {
            "text": ref if isinstance(ref, list) else [ref],
            "answer_start": [0] * len(ref if isinstance(ref, list) else [ref])
        }
    }
    for i, ref in enumerate(dataset["answer"])
]

# 计算 SQuAD-style EM 和 F1
results = squad_metric.compute(predictions=predictions, references=references)

# 打印平均指标
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")

Exact Match: 24.55
F1 Score: 37.08


In [18]:
squad_scored_UND, UND_f1_list, UND_em_list = evaluate_squad_per_sample_multi_ref_pred(underspecified_set_qa, ref_col="answer")
squad_scored_UND.to_json("./intermediate/BASELINE_frames_UND_qa_Gemini_with_squad_scores.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 92.60ba/s]


1964604

In [19]:
squad_scored_FS, FS_f1_list, FS_em_list = evaluate_squad_per_sample_multi_ref_pred(fully_specified_set_qa, ref_col="answer")
squad_scored_FS.to_json("./intermediate/BASELINE_frames_FS_qa_Gemini_with_squad_scores.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 111.20ba/s]


1678362

In [20]:
df = pd.read_json("./intermediate/BASELINE_frames_UND_qa_Gemini_with_squad_scores.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_frames_UND_qa_Gemini_with_squad_scores.csv')

df = pd.read_json("./intermediate/BASELINE_frames_FS_qa_Gemini_with_squad_scores.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_frames_FS_qa_Gemini_with_squad_scores.csv')

In [21]:
UND_mean_em = np.mean(UND_em_list)  # em_scores: EM list per sample
UND_mean_f1 = np.mean(UND_f1_list)  # f1_scores F1 list per sample
print(f"UND Exact Match (avg): {UND_mean_em * 100:.2f}")
print(f"UND F1 Score (avg): {UND_mean_f1 * 100:.2f}")

FS_mean_em = np.mean(FS_em_list)  # em_scores: EM list per sample
FS_mean_f1 = np.mean(FS_f1_list)  # f1_scores F1 list per sample
print(f"FS Exact Match (avg): {FS_mean_em * 100:.2f}")
print(f"FS F1 Score (avg): {FS_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(FS_f1_list, UND_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(FS_em_list, UND_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

UND Exact Match (avg): 24.55
UND F1 Score (avg): 37.08
FS Exact Match (avg): 41.41
FS F1 Score (avg): 57.46
F1: t=6.990, p=0.0000
EM: t=5.190, p=0.0000


### Ragas

In [22]:
from helper_functions_qa import answer_accuracy

In [23]:
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

UND_full = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_frames_UND_qa_Gemini_with_squad_scores.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

FS_full = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_frames_FS_qa_Gemini_with_squad_scores.jsonl",
    split="all"
)

Generating train split: 440 examples [00:00, 71993.98 examples/s]
Generating train split: 384 examples [00:00, 58170.06 examples/s]


In [24]:
UND_ragas = await answer_accuracy(UND_full, evaluator_llm, ref_col = "answer")
UND_ragas.to_csv("./output_csv/frames_UND_Gemini_Ragas.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 29.50ba/s]


1900275

In [25]:
FS_ragas = await answer_accuracy(FS_full, evaluator_llm, ref_col = "answer")
FS_ragas.to_csv("./output_csv/frames_FS_Gemini_Ragas.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 35.08ba/s]


1621133

In [26]:
UND_ragas_AA = list(UND_ragas["ragas_AA_short"])
FS_ragas_AA = list(FS_ragas["ragas_AA_short"])

UND_mean_AA = np.mean(UND_ragas_AA)
print(f"UND AA (avg): {UND_mean_AA * 100:.2f}")


FS_mean_AA = np.mean(FS_ragas_AA)
print(f"FS AA (avg): {FS_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(FS_ragas_AA, UND_ragas_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

UND AA (avg): 42.05
FS AA (avg): 69.47
AA: t=8.609, p=0.0000
